In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model

import numpy as np

import glob
import os
import sys
import cv2

# import datetime
from PIL import Image

from matplotlib import *
from matplotlib import pyplot
from matplotlib.pyplot import *

from skimage import metrics
from skimage.metrics import structural_similarity as ssim
import seaborn as sns


In [2]:
NOISE_FLOOR = 30
BATCH_SIZE = 16
LATENT_DIM = 100
IMAGE_SIZE = 128
G_LEARNING_RATE = 0.0002
D_LEARNING_RATE = 0.0001
G_DECAY = 0.6
D_DECAY = 0.6
EPOCHS = 5001
DISPLAY_SAMPLES = 10

In [3]:
model_dir = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Code_v02/Models/Pit_RGB_generator_model_2000.h5"
results = "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Brain MRI images Classification/Code_v02/Images/"


In [4]:
generator = load_model(model_dir)

: 

: 

In [ ]:
def reduce_noise(noisy_img, noise_floor=NOISE_FLOOR):
    clean_img = np.array(noisy_img)
    clean_img[noisy_img < NOISE_FLOOR] = 0
    return clean_img


In [ ]:
def import_data():
    # check current directory and import images
    print(os.getcwd())
    filelist = glob.glob(
        "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/Dataset/train/meningioma/*.jpg"
    )
    train_size = len(filelist)
    # images0=np.array([np.array( ) for i in filelist[0:train_size]])
    images0 = []
    for i in filelist[0:train_size]:
        image = Image.open(i)
        image = np.array(image, dtype="float32")
        # image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = reduce_noise(image)
        image = (image - np.min(image)) / (np.max(image) - np.min(image))
        # Rescale the pixel values to be between -1 and 1
        image = (image * 2) - 1
        image = cv2.resize(image, (128, 128))
        images0.append(image)
    images0 = np.array(images0)
    print("training images", images0.shape)

    filelist = glob.glob(
        "C:/Users/alime/Dropbox/PC/Documents/Coding/Memoire 2023/Datasets/Brain-Tumor-Dataset-RAW/Cropped_Image_Sets/Dataset/val/meningioma/*.jpg"
    )
    test_size = len(filelist)
    images1 = []
    for i in filelist[0:test_size]:
        image = Image.open(i)
        image = np.array(image, dtype="float32")
        # image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = reduce_noise(image)
        image = (image - np.min(image)) / (np.max(image) - np.min(image))
        # Rescale the pixel values to be between -1 and 1
        image = (image * 2) - 1
        image = cv2.resize(image, (128, 128))
        images1.append(image)
    images1 = np.array(images1)
    # print("training images", images1.shape)
    images = np.concatenate((images0, images1), axis=0)
    print("Dataset Shape: ", images.shape)

    # make into 4D array
    images = images[:, :, :, np.newaxis]

    # check shape
    print(images.shape)
    return images

# Import Generator

In [ ]:
def generate_images(generator_path):
    noise = tf.random.normal([DISPLAY_SAMPLES, LATENT_DIM]) # Create Noise
    generated = generator(noise, training=False)
    print("-------------- Generating -------------------")
    print("Generated Images Shape: ", generated.shape)
    return generated
generated= generate_images(model_dir)

# Display images

In [ ]:
def display_save(filename):
    # ex: filename = 'menin_4000_RGB_generated_03.png'
    figure_name = f"{results}{filename}"
    # pyplot.figure(figsize=(2, 5))
    for i in range(DISPLAY_SAMPLES):
        # define subplot
        pyplot.subplot(4, 10, 1 + i)
        pyplot.axis("off")
        # plot single image
        pyplot.imshow(generated[i, :, :, 0], cmap="gray")
        pyplot.savefig(
            figure_name,
            dpi=300,
            transparent=True,
        )
    pyplot.show()
    pyplot.close()

    # images = (images - 127.5) / 127.5
    # test_images = images[:DISPLAY_SAMPLES]
    # print("images: ", test_images.shape)


# Caclculate SSIM structural metric

In [ ]:

# since calculating SSIM for one image is computationally expensive, just choose the index of one image to calculate
# whichfake is the index of the sample image
images = import_data()
test_images = images[:DISPLAY_SAMPLES]
print("images: ", test_images.shape)


def SSIM():
    
    whichfake = 4
    
    # create array to store SSIM values
    ssim_noise = []

    # calculate SSIM for each training image
    for i in range(images.shape[0]):
        ssim_noise.append(
            ssim(
                images[i, :, :, 0],
                generated.numpy()[whichfake, :, :, 0],
                data_range=np.max(generated.numpy()[whichfake, :, :, 0])
                - np.min(generated.numpy()[whichfake, :, :, 0]),
            )
        )

    fig_1 = f"{results}menin_4000_RGB_SSIM_03_{np.max(ssim_noise)}.png"
    # plot generated image and OASIS image that corresponds to the highest SSIM value
    fig, axs = pyplot.subplots(2, 1, constrained_layout=True, figsize=(10, 10))
    axs[0].imshow(generated[whichfake, :, :, 0], cmap="gray")
    axs[0].set_title("Generated image with max SSIM: {:.4f}".format(np.max(ssim_noise)))
    # pyplot.savefig("/kaggle/working/Generated {}.png".format(np.max(ssim_noise)))
    pyplot.savefig(
        fig_1,
        bbox_inches="tight",
        transparent=True,
    )
    print("Generated image SSIM: ", np.max(ssim_noise))

    fig_2 = f"{results}menin_2000_RGB_SSIM_03_{np.max(ssim_noise)}.png"

    axs[1].imshow(images[ssim_noise.index(np.max(ssim_noise)), :, :, 0], cmap="gray")
    axs[1].set_title(
        "Closest Pituitary image {:.0f}".format(ssim_noise.index(np.max(ssim_noise)))
    )
    pyplot.savefig(
        fig_2,
        bbox_inches="tight",
        transparent=True,
    )
    print("Closest image SSIM: ", np.max(ssim_noise))
    pyplot.show()

# Visualize Distribution

In [ ]:
def visualize_ditribution():
    distribution = f"{results}menin_4000_RGB_Distribution_03.png"
    fig, axs = pyplot.subplots(ncols=1, nrows=1, figsize=(18, 10))

    sns.distplot(test_images, label="Real Images", hist=True, color="#fc0328", ax=axs)
    sns.distplot(generated, label="Generated Images", hist=True, color="#0c06c7", ax=axs)

    axs.legend(loc="upper right", prop={"size": 12})

    pyplot.savefig(
        distribution,
        bbox_inches="tight",
        transparent=True,
    )
    pyplot.show()
    pyplot.close()
